# 11 – Deskriptive Statistik für den Anhang (Training FY2012–2017)

Dieses Notebook **schätzt kein Modell**. Es beschreibt die Trainingsstichprobe so, wie sie **vor** dem Kappen (Winsorisieren) und Auffüllen (Imputation) aus Notebook 04/06 aussah:

1. Für alle 16 Variablen von Modell B: N, Mittelwert, Standardabweichung, P25, Median, P75, getrennt für Ziele und Nicht-Ziele.
2. Eine Spearman-Korrelationsmatrix dieser 16 Variablen.

Warum Rohwerte? Die Tabelle soll zeigen, wie die Daten tatsächlich aussehen. Nach dem Auffüllen stünde bei einem Drittel der Firmen beim Umsatzwachstum einfach der Median – das würde Mittelwerte und Streuung verfälschen. **N** zählt deshalb nur die Zeilen, in denen ein Wert vorhanden ist; die Lücke zu allen Zeilen zeigt, wie viel fehlt.

Die Fehlend-Indikatoren (`*_missing`) stehen nicht in den Tabellen: Ihr Inhalt steckt bereits in der Spalte N.

Ergebnis: `thesis_tables/appendix_descriptives.csv/.md` und `thesis_tables/appendix_correlations.csv/.md`. Daten und Pipeline werden nur gelesen.

## 1. Rohwerte bauen (gleicher Code wie Notebook 04, Abschnitt 1–2)

Stichprobenfilter aus Notebook 02 und die Palepu-Variablen wie in 04 – kompakt wiederholt, weil 04 nur die aufbereiteten Werte speichert. Die sieben neuen Variablen von Modell B kommen als Rohwerte aus `firm_year_ratios.csv`, genau wie in Notebook 06.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("..") / "data"
TAB = Path("..") / "thesis_tables"
Y = "y_pe_main_h3"

panel = pd.read_csv(DATA / "processed" / "firm_year_panel.csv", dtype={"cik": str})
ratios = pd.read_csv(DATA / "processed" / "firm_year_ratios.csv", dtype={"cik": str})
fund = pd.read_csv(DATA / "processed" / "firm_year_fundamentals.csv", dtype={"cik": str})

B_NEW = ["fcf_to_assets", "fcf_vol", "interest_coverage", "ppe_to_assets", "rd_to_assets", "cash_ratio", "altman_z2"]
ratio_cols = ["sales_growth_1y", "leverage", "log_assets", "float_to_book", "float_to_earnings"] + B_NEW
fund_cols = ["net_income", "equity", "cash", "liabilities_current", "total_assets"]
df = (panel.merge(ratios[["accession"] + ratio_cols], on="accession", validate="one_to_one")
           .merge(fund[["accession"] + fund_cols], on="accession", validate="one_to_one"))

# Stichprobenfilter (Notebook 02)
s = df[df["h3_observed"]]
s = s[~s["fin_sic_60_64"].astype(bool)]
s = s[s["completion_status"] != "uncertain"]
s = s[~pd.to_numeric(s["sic"], errors="coerce").isin([6770])]
s = s[(s["entity_public_float"] >= 50e6) | s["first_listing_year_exempt"].astype(bool)]
sample = s.copy()

# roe und liquidity (Notebook 03)
sample["roe"] = sample["net_income"] / sample["equity"].where(sample["equity"] > 0)
sample["liquidity"] = (sample["cash"] - sample["liabilities_current"]) / sample["total_assets"].where(sample["total_assets"] > 0)

# grdummy: hoch/niedrig relativ zum Median des Geschäftsjahres (Notebook 03); fehlt, wenn eine Zutat fehlt
def high_vs_year_median(col):
    high = sample[col] > sample.groupby("fiscal_year")[col].transform("median")
    return high.where(sample[col].notna())

g, l, v = (high_vs_year_median(c) for c in ["sales_growth_1y", "liquidity", "leverage"])
combo = ((g == False) & (l == True) & (v == False)) | ((g == True) & (l == False) & (v == True))
sample["grdummy"] = combo.astype(float).where(g.notna() & l.notna() & v.notna())

# idummy: pe_main-Event in derselben 2-stelligen SIC im Kalenderjahr davor (Notebook 03)
events = pd.read_csv(DATA / "interim" / "combined_events.csv", dtype={"cik": str})
events = events[events["sponsor_like_ptp"].astype(str) == "True"].copy()
events["event_date"] = pd.to_datetime(events["first_event_date"])
events["event_year"] = events["event_date"].dt.year
filings = pd.read_csv(DATA / "interim" / "stage_11_raw_quarters.csv", dtype={"cik": str},
                      usecols=["cik", "sic", "filed_date"]).dropna(subset=["sic"])
filings["filed_date"] = pd.to_datetime(filings["filed_date"])
f = filings.merge(events[["cik", "event_date"]], on="cik")
f["before"] = f["filed_date"] <= f["event_date"]
f["gap"] = (f["filed_date"] - f["event_date"]).abs()
f = pd.concat([f[f["before"]].sort_values("filed_date", ascending=False).groupby("cik").head(1),
               f[~f["before"]].sort_values("gap").groupby("cik").head(1)]).drop_duplicates("cik")
events = events.merge(f[["cik", "sic"]], on="cik", how="left")
events["sic2"] = events["sic"] // 100
active = events.dropna(subset=["sic2"]).groupby(["sic2", "event_year"]).size().rename("n_events").reset_index()
sample["sic2"] = pd.to_numeric(sample["sic"], errors="coerce") // 100
sample["lookup_year"] = sample["fiscal_year"] - 1
sample = sample.merge(active.rename(columns={"event_year": "lookup_year"}), on=["sic2", "lookup_year"], how="left")
sample["idummy"] = (sample["n_events"].fillna(0) > 0).astype(float).where(sample["sic2"].notna())

# Nur das Training (FY2012-2017)
raw = sample[sample["fiscal_year"] <= 2017].copy()
print(f"Training: {len(raw):,} Firmenjahre, davon {int(raw[Y].sum())} Ziele und {int((raw[Y] == 0).sum()):,} Nicht-Ziele")

Training: 18,058 Firmenjahre, davon 167 Ziele und 17,891 Nicht-Ziele


**Kontrolle:** Kappt man diese Rohwerte mit den Trainingsgrenzen und füllt mit dem Trainingsmedian auf (Regeln aus 04), müssen genau die Werte aus `model_data_palepu.csv` herauskommen. Dann sind es sicher dieselben Zeilen und dieselben Rohwerte.

In [2]:
model_data = pd.read_csv(DATA / "processed" / "model_data_palepu.csv", dtype={"cik": str})
md_train = model_data[model_data["split"] == "train"].set_index("accession")
r = raw.set_index("accession").loc[md_train.index]
assert len(r) == len(md_train) == 18_058 and r[Y].sum() == md_train[Y].sum() == 167

continuous = ["roe", "sales_growth_1y", "liquidity", "leverage", "log_assets", "float_to_book", "float_to_earnings"]
lo, hi = r[continuous].quantile(0.01), r[continuous].quantile(0.99)
med = r[continuous].clip(lo, hi, axis=1).median()
rebuilt = r[continuous].clip(lo, hi, axis=1).fillna(med)
assert np.allclose(rebuilt.to_numpy(), md_train[continuous].to_numpy())
assert (r["grdummy"].fillna(0) == md_train["grdummy"]).all()
assert (r["idummy"].fillna(r["idummy"].median()) == md_train["idummy"]).all()
print("Kontrolle bestanden: Rohwerte reproduzieren model_data_palepu.csv (Training) exakt.")

Kontrolle bestanden: Rohwerte reproduzieren model_data_palepu.csv (Training) exakt.


## 2. Deskriptive Statistik: Ziele vs. Nicht-Ziele

In [3]:
B_VARS = ["roe", "sales_growth_1y", "liquidity", "leverage", "grdummy", "idummy", "log_assets",
          "float_to_book", "float_to_earnings"] + B_NEW
LABELS = {
    "roe": "Return on equity", "sales_growth_1y": "Sales growth (1y)", "liquidity": "Liquidity",
    "leverage": "Leverage", "grdummy": "Growth–resource imbalance", "idummy": "Industry PE activity",
    "log_assets": "Log total assets", "float_to_book": "Float / book", "float_to_earnings": "Float / earnings",
    "fcf_to_assets": "Free cash flow / assets", "fcf_vol": "Cash flow volatility", "interest_coverage": "Interest coverage",
    "ppe_to_assets": "PP&E / assets", "rd_to_assets": "R&D / assets", "cash_ratio": "Cash ratio",
    "altman_z2": "Altman Z''-score",
}
GROUPS = {"Targets": raw[Y] == 1, "Non-targets": raw[Y] == 0}

rows = []
for v in B_VARS:
    for group, mask in GROUPS.items():
        x = raw.loc[mask, v].dropna()                    # nur vorhandene Werte
        rows.append({"variable": LABELS[v], "code": v, "group": group, "N": len(x),
                     "N total": int(mask.sum()), "mean": x.mean(), "SD": x.std(),
                     "P25": x.quantile(0.25), "median": x.median(), "P75": x.quantile(0.75)})
desc = pd.DataFrame(rows)
desc

,variable,code,group,N,N total,mean,SD,P25,median,P75
0,Return on equity,roe,Targets,138,167,-0.081687,1.131598e+00,-0.057457,0.049577,0.129505
1,Return on equity,roe,Non-targets,15716,17891,-0.457706,1.815720e+01,-0.078528,0.069405,0.149688
2,Sales growth (1y),sales_growth_1y,Targets,130,167,0.147383,3.888523e-01,-0.003476,0.075903,0.208041
3,Sales growth (1y),sales_growth_1y,Non-targets,12239,17891,1.035377,3.098071e+01,-0.024640,0.054475,0.166853
4,Liquidity,liquidity,Targets,149,167,-0.086905,1.897660e-01,-0.165732,-0.085032,-0.013260
5,Liquidity,liquidity,Non-targets,15708,17891,-23.443955,1.938512e+03,-0.171769,-0.076312,0.027139
6,Leverage,leverage,Targets,163,167,0.264563,2.375785e-01,0.056921,0.232583,0.416102
7,Leverage,leverage,Non-targets,17467,17891,30.774068,3.131264e+03,0.005974,0.207748,0.386001
8,Growth–resource imbalance,grdummy,Targets,117,167,0.136752,3.450633e-01,0.000000,0.000000,0.000000
9,Growth–resource imbalance,grdummy,Non-targets,10883,17891,0.265827,4.417931e-01,0.000000,0.000000,1.000000


**Ergebnis:** Bei den Medianen unterscheiden sich Ziele und Nicht-Ziele vor allem bei der Zinsdeckung (2,6 statt 3,1), beim Altman-Z'' (2,7 statt 4,1) und bei Float/Buchwert (1,6 statt 1,9). Ziele sind also etwas niedriger bewertet und näher am Distress-Bereich, haben aber leicht höheren freien Cashflow (0,050 statt 0,035). Mittelwerte und Standardabweichungen der Nicht-Ziele sind bei mehreren Kennzahlen riesig (z. B. Float/Buchwert), weil einzelne Firmen mit fast leerer Bilanz extreme Quotienten haben. Für den Text sind deshalb Median und Quartile die aussagekräftigen Zahlen. Wachstum-Ressourcen-Ungleichgewicht (`grdummy`) ist bei Zielen seltener (14 % statt 27 %), PE-Aktivität in der Branche häufiger (33 % statt 26 %).

## 3. Spearman-Korrelationen (Training, Rohwerte)

Spearman korreliert die Ränge, ist also robust gegen die Extremwerte, die in den Rohwerten noch stecken. Jedes Paar wird auf den Zeilen berechnet, in denen beide Werte vorhanden sind (paarweise).

In [4]:
spear_raw = raw[B_VARS].corr(method="spearman")
pairs_n = raw[B_VARS].notna().astype(int).T @ raw[B_VARS].notna().astype(int)   # Zeilen pro Paar
print(f"Kleinste Paar-Stichprobe: {pairs_n.to_numpy().min():,} Zeilen")
strong = [(a, b, round(spear_raw.loc[a, b], 3)) for i, a in enumerate(B_VARS) for b in B_VARS[i + 1:]
          if abs(spear_raw.loc[a, b]) > 0.5]
print("Paare mit |rho| > 0.5:", strong)
print(f"ROE / Zinsdeckung: rho = {spear_raw.loc['roe', 'interest_coverage']:.3f} "
      f"auf {pairs_n.loc['roe', 'interest_coverage']:,} Zeilen mit beiden Werten")
spear_raw.round(2)

Kleinste Paar-Stichprobe: 8,174 Zeilen
Paare mit |rho| > 0.5: [('roe', 'fcf_to_assets', np.float64(0.633)), ('roe', 'interest_coverage', np.float64(0.733)), ('liquidity', 'cash_ratio', np.float64(0.533)), ('log_assets', 'fcf_vol', np.float64(-0.516)), ('float_to_earnings', 'interest_coverage', np.float64(0.568)), ('fcf_to_assets', 'interest_coverage', np.float64(0.61)), ('interest_coverage', 'altman_z2', np.float64(0.55)), ('rd_to_assets', 'cash_ratio', np.float64(0.522))]
ROE / Zinsdeckung: rho = 0.733 auf 11,045 Zeilen mit beiden Werten


,roe,sales_growth_1y,liquidity,leverage,grdummy,idummy,log_assets,float_to_book,float_to_earnings,fcf_to_assets,fcf_vol,interest_coverage,ppe_to_assets,rd_to_assets,cash_ratio,altman_z2
roe,1.00,0.04,-0.25,0.16,-0.03,-0.03,0.43,0.15,0.48,0.63,-0.30,0.73,0.16,-0.35,-0.20,0.30
sales_growth_1y,0.04,1.00,0.05,-0.02,-0.05,0.09,-0.06,0.13,0.06,0.02,0.03,0.06,-0.08,0.09,0.01,0.08
liquidity,-0.25,0.05,1.00,-0.29,0.08,0.02,-0.19,0.07,-0.09,-0.06,0.16,-0.07,-0.16,0.36,0.53,0.29
leverage,0.16,-0.02,-0.29,1.00,-0.04,-0.06,0.39,-0.11,0.07,0.03,-0.21,-0.19,0.30,-0.32,-0.43,-0.47
grdummy,-0.03,-0.05,0.08,-0.04,1.00,0.00,-0.07,-0.05,-0.01,-0.03,0.05,-0.01,-0.02,0.02,0.01,0.04
idummy,-0.03,0.09,0.02,-0.06,0.00,1.00,-0.03,0.06,0.02,0.08,-0.03,-0.02,-0.12,0.11,0.08,0.04
log_assets,0.43,-0.06,-0.19,0.39,-0.07,-0.03,1.00,-0.02,0.31,0.36,-0.52,0.28,0.32,-0.35,-0.44,-0.06
float_to_book,0.15,0.13,0.07,-0.11,-0.05,0.06,-0.02,1.00,0.15,0.15,0.02,0.11,-0.12,0.22,0.19,0.28
float_to_earnings,0.48,0.06,-0.09,0.07,-0.01,0.02,0.31,0.15,1.00,0.42,-0.30,0.57,0.14,-0.22,-0.18,0.26
fcf_to_assets,0.63,0.02,-0.06,0.03,-0.03,0.08,0.36,0.15,0.42,1.00,-0.29,0.61,0.10,-0.22,-0.09,0.36


**Ergebnis:** Die meisten Paare sind schwach korreliert. Am stärksten hängen Eigenkapitalrendite und Zinsdeckung zusammen (rho 0,73), gefolgt von Eigenkapitalrendite und freiem Cashflow (0,63). Profitable Firmen haben eben auch hohen Cashflow und decken ihre Zinsen leicht. Wichtig: ROE/Zinsdeckung liegt hier **über** der Schwelle 0,7, in Notebook 06 aber darunter (0,58). Dort wurde auf den aufbereiteten Daten gerechnet, in denen die fehlende Zinsdeckung (32 %) mit dem Median gefüllt ist. Diese gefüllten Werte schwächen den Zusammenhang ab. Hier zählen nur die Zeilen, in denen beide Werte vorhanden sind. Die Variablenauswahl aus 06 bleibt unverändert.

## 4. Speichern (CSV mit voller Genauigkeit, Markdown gerundet)

In [5]:
def md_table(df):
    """DataFrame -> Markdown-Tabelle (ohne Zusatzpaket), wie in Notebook 10."""
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join("" if (not isinstance(v, str) and pd.isna(v)) else str(v) for v in r) + " |")
    return "\n".join(lines)

def write_table(df, name, title, notes=(), md_df=None):
    """Speichert die Tabelle als CSV (volle Genauigkeit) und als Markdown (gerundet, mit Titel und Anmerkungen)."""
    df.to_csv(TAB / f"{name}.csv", index=False)
    text = f"# {title}\n\n" + md_table(md_df if md_df is not None else df)
    if notes:
        text += "\n\n" + "\n".join(f"- {n}" for n in notes)
    (TAB / f"{name}.md").write_text(text + "\n", encoding="utf-8")
    print(f"geschrieben: thesis_tables/{name}.csv und .md ({len(df)} Zeilen)")

# Große Werte (z. B. Zinsdeckung) mit einer, alle übrigen mit drei Nachkommastellen
num = lambda x: f"{x:,.1f}" if abs(x) >= 100 else f"{x:.3f}"
desc_md = desc.drop(columns="code").copy()
for c in ["mean", "SD", "P25", "median", "P75"]:
    desc_md[c] = desc_md[c].map(num)
desc_md["N"] = desc_md["N"].map("{:,}".format)
desc_md["N total"] = desc_md["N total"].map("{:,}".format)
write_table(desc, "appendix_descriptives",
            "Descriptive statistics, training sample FY2012–2017 (raw values before winsorising and imputation)",
            notes=[f"Training sample after the sample filters: {len(raw):,} firm-years, {int(raw[Y].sum())} targets "
                   f"(PE take-private within 3 years, pe_main) and {int((raw[Y] == 0).sum()):,} non-targets.",
                   "N = firm-years with a non-missing value; N total = all firm-years in the group.",
                   "Growth–resource imbalance and industry PE activity are 0/1 variables: the mean is the share of ones.",
                   "Raw values include extreme ratios of firms with very small total assets, so means and SDs are "
                   "driven by outliers; medians and quartiles are robust. The models use values winsorised at the "
                   "training P1/P99 (Notebooks 04 and 06)."],
            md_df=desc_md)

corr_out = spear_raw.copy()
corr_out.index = [LABELS[v] for v in B_VARS]
corr_out.columns = [LABELS[v] for v in B_VARS]
corr_md = corr_out.round(2).map(lambda x: f"{x:.2f}")
corr_md.insert(0, "variable", corr_md.index)
write_table(corr_out.rename_axis("variable").reset_index(), "appendix_correlations",
            "Spearman rank correlations of the Model B variables, training sample FY2012–2017 (raw values)",
            notes=[f"Pairwise complete observations; the smallest pair sample has {pairs_n.to_numpy().min():,} firm-years.",
                   "Spearman correlations are computed on ranks and are therefore robust to the extreme raw values.",
                   "Variable selection in Notebook 06 used |rho| > 0.7 on the prepared (winsorised and imputed) "
                   "training data; return on assets and log public float were dropped there and are not shown.",
                   f"Return on equity / interest coverage: rho = {spear_raw.loc['roe', 'interest_coverage']:.2f} on the "
                   f"{pairs_n.loc['roe', 'interest_coverage']:,} firm-years with both values, but 0.58 on the prepared "
                   "data used for the selection in Notebook 06 (median-filled missing coverage weakens the association)."],
            md_df=corr_md)

geschrieben: thesis_tables/appendix_descriptives.csv und .md (32 Zeilen)
geschrieben: thesis_tables/appendix_correlations.csv und .md (16 Zeilen)
